<a href="https://colab.research.google.com/github/cbonnin88/EcoFlux/blob/main/Ecoflux_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df_events = pl.read_csv('ecoflux_amplitude_events.csv')

# **2. Transformation: Create Event Types and Calculate Carbon Impact**

In [4]:
# I used vectorized logic to define 'App Open' vs 'Smart Charge'
processed_df = df_events.with_columns([
    pl.from_epoch(pl.col('time'), time_unit='ms').alias('timestamp'),
    pl.when(pl.col('activity_score') > 0.85).then(pl.lit('Schedule Smart Charge'))
      .when(pl.col('activity_score') > 0.30).then(pl.lit('App Open'))
      .otherwise(pl.lit('Background')).alias('event_type')
]).with_columns([
    # Calculate CO2 impact only for Smart Charge events
    pl.when(pl.col('event_type') == 'Schedule Smart Charge')
    .then(pl.col('activity_score')* 12.5)
    .otherwise(0.0).alias('co2_saved_kg')
])

# **3. Aggregate: Daily Active Users (DAU) by Country**

In [7]:
dau_summary = (
    processed_df.group_by(['country','event_date'])
    .agg(pl.col('user_id').n_unique().alias('dau'))
    .sort('event_date')
)

display(dau_summary.head())

country,event_date,dau
str,str,u32
"""Germany""","""2026-02-01T00:00:00.000000""",55
"""Spain""","""2026-02-01T00:00:00.000000""",45
"""Luxembourg""","""2026-02-01T00:00:00.000000""",39
"""France""","""2026-02-01T00:00:00.000000""",57
"""United Kingdom""","""2026-02-01T00:00:00.000000""",54


# **4. Aggregate: Total Impact by Subscription Plan**

In [9]:
plan_impact = (
    processed_df.group_by(['country','subscription_plan'])
    .agg(pl.col('co2_saved_kg').round(1).sum().alias('total_co2'))
    .sort('total_co2',descending=True)
)

display(plan_impact.head())

country,subscription_plan,total_co2
str,str,f64
"""France""","""Basic""",17163.9
"""United Kingdom""","""Basic""",16030.1
"""Denmark""","""Basic""",15384.7
"""Germany""","""Basic""",14993.3
"""Spain""","""Basic""",14721.0


# **The DAU Trend**
- Tracking Adoption across our 6 Markets

In [11]:
fig_dau = px.line(
    dau_summary.to_pandas(),
    x='event_date',
    y='dau',
    color='country',
    title='EcoFlux: Daily Active users (DAU) by Country',
    labels= {'dau':'Active Users','event_date':'Date','country':'Countries'}
)

fig_dau.show()

# **Sustainability Impact**
- Shows which country and which plan is driving the most environmental impact

In [25]:
fig_impact = px.bar(
    plan_impact.to_pandas(),
    x='subscription_plan',
    y='total_co2',
    color='country',
    title='Total Carbon Displacement by Market & Tier',
    barmode='stack',
    labels={'total_co2':'CO2 Saved (kg)','country':'Country','subscription_plan':'Subscription'},
    color_discrete_sequence=px.colors.sequential.Viridis_r
)

fig_impact.show()

# **Retention Funnel**

In [18]:
funnel_data = processed_df.select([
    pl.col('event_type').value_counts()
]).unnest('event_type')

In [19]:
# Filtering for two main funnel steps
steps = ['App Open','Schedule Smart Charge']
counts = [
    funnel_data.filter(pl.col('event_type')== s)['count'][0] for s in steps
]

In [23]:
fig_funnel = go.Figure(go.Funnel(
    y=steps,
    x=counts,
    textinfo='value+percent initial',
    marker={'color':['#636EFA','#00CC96']}
))

fig_funnel.update_layout(title_text='Product Conversion Funnel')
fig_funnel.show()